In [ ]:
# Environment Setup & Raw Sample Generation

import os
import numpy as np
import pandas as pd

raw_dir = 'data/raw'
processed_dir = 'data/processed'

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

# Generate sample dataset containing missing values
data = {
    'age': [34, 45, 29, 50, 38, np.nan, 41],
    'income': [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    'score': [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    'zipcode': ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
    'city': ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco'],
    'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan]
}

df_raw = pd.DataFrame(data)
csv_path = os.path.join(raw_dir, 'sample_data.csv')
df_raw.to_csv(csv_path, index=False)
print(f"Raw dataset created at: {csv_path}")

Raw dataset created at: data/raw\sample_data.csv


In [ ]:
# Import Custom Preprocessing Module

import sys
from pathlib import Path

# Add project root to sys.path to enable src imports
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import cleaning

print("Successfully loaded src.cleaning module.")

Successfully loaded src.cleaning module.


In [ ]:
# Load & Inspect Raw Data

df = pd.read_csv('data/raw/sample_data.csv')

print("--- Raw Dataset Preview ---")
print(df.head())
print("\n--- Missing Values Per Column ---")
print(df.isna().sum())

--- Raw Dataset Preview ---
    age   income  score  zipcode      city  extra_data
0  34.0  55000.0   0.82    90210   Beverly         NaN
1  45.0      NaN   0.91    10001  New York        42.0
2  29.0  42000.0    NaN    60614   Chicago         NaN
3  50.0  58000.0   0.76    94103        SF         NaN
4  38.0      NaN   0.88    73301    Austin         NaN

--- Missing Values Per Column ---
age           1
income        3
score         1
zipcode       0
city          0
extra_data    5
dtype: int64


In [5]:
# Apply Sequential Cleaning Pipeline

df_cleaned = df.copy()

# Step 1: Drop sparse columns with >50% missing values (e.g., extra_data)
df_cleaned = cleaning.drop_missing(df_cleaned, threshold=0.5)

# Step 2: Impute numeric features using median
df_cleaned = cleaning.fill_missing_median(df_cleaned, columns=['age', 'income', 'score'])

# Step 3: Normalize numeric feature distributions to range [0, 1]
df_cleaned = cleaning.normalize_data(df_cleaned, columns=['age', 'income', 'score'], method='minmax')

print("--- Cleaned Dataset Preview ---")
print(df_cleaned.head())

--- Cleaned Dataset Preview ---
        age  income     score  zipcode      city
0  0.238095  0.8125  0.653846    90210   Beverly
1  0.761905  0.6250  1.000000    10001  New York
2  0.000000  0.0000  0.596154    60614   Chicago
3  1.000000  1.0000  0.423077    94103        SF
4  0.428571  0.6250  0.884615    73301    Austin


In [6]:
# Save Cleaned Dataset & Verification

output_path = 'data/processed/sample_data_cleaned.csv'
df_cleaned.to_csv(output_path, index=False)
print(f"Saved processed dataset to: {output_path}")

# Pipeline Comparative Verification
print("\n--- Dataset Comparison Summary ---")
print(f"Original Shape : {df.shape}")
print(f"Cleaned Shape  : {df_cleaned.shape}")
print(f"Original NaNs  : {df.isna().sum().sum()}")
print(f"Cleaned NaNs   : {df_cleaned.isna().sum().sum()}")

Saved processed dataset to: data/processed/sample_data_cleaned.csv

--- Dataset Comparison Summary ---
Original Shape : (7, 6)
Cleaned Shape  : (7, 5)
Original NaNs  : 10
Cleaned NaNs   : 0



# Documentation of Assumptions & Tradeoffs

## Preprocessing Analysis & Tradeoffs

### 1. Column Dropping Threshold
- **Action**: Columns with >50% missing values (`extra_data`) were dropped.
- **Rationale**: `extra_data` contained ~71% NaNs. Imputing majority missing columns introduces heavy artificial noise.

### 2. Median Imputation
- **Action**: Applied median imputation to `age`, `income`, and `score`.
- **Rationale**: Median imputation is robust to skewness and extreme outliers compared to mean imputation.

### 3. Min-Max Normalization
- **Action**: Scaled `age`, `income`, and `score` into $[0, 1]$.
- **Rationale**: Standardizes scale ranges for distance-based downstream modeling algorithms while preserving original underlying feature distributions.